In [1]:
from readability import Readability
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
import pandas as pd

In [69]:
dataset = 'nq_test'
retriever = 'bm25'
retr_res = pd.read_csv(f'../../rag_utility/res/{retriever}_{dataset}.csv')

In [70]:
import pyterrier as pt

if not pt.java.started():
    pt.java.init()

In [71]:
if('nq' in dataset):
    index = pt.Artifact.from_hf('pyterrier/ragwiki-terrier')
else:
    index_path ='/mnt/indices/msmarco-passage.terrier/'
    index_ref = pt.IndexRef.of(index_path)
    index = pt.IndexFactory.of(index_ref)

# index_path ='/mnt/indices/msmarco-passage.terrier/'
# index_ref = pt.IndexRef.of(index_path)
# index = pt.IndexFactory.of(index_ref)

In [72]:
text_loader = index.text_loader(["text"])

In [73]:
def calculate_readability(_input_text: str, _qid: str, _docno: str):
    r = Readability(_input_text)
    # Map names to the corresponding method calls (lambdas delay execution)
    metrics = {
        "Dale Chall": lambda: r.dale_chall(),
        "Spache": lambda: r.spache(),
        "Flesch-Kincaid": lambda: r.flesch_kincaid(),
        "Flesch": lambda: r.flesch(),
        "Gunning Fog": lambda: r.gunning_fog(),
        "Coleman Liau": lambda: r.coleman_liau(),
        "ARI": lambda: r.ari(),
        "Linsear Write": lambda: r.linsear_write(),
        "SMOG": lambda: r.smog(),
    }
    
    to_output = []
    for name, func in metrics.items():
        try:
            readability_result = func()
            score = readability_result.score
            try:
                grade_level = readability_result.grade_levels
            except:
                grade_level = []
            to_output.append([_qid, _docno, name, score, grade_level])
        except Exception as e:
            to_output.append([_qid, _docno, name, -1, []])
            # print(f"No {name}: {e}")

    # if(len(to_output)==0):
    #     print(_qid)
    return to_output

In [79]:
from tqdm import tqdm
import pathlib

_k = 2

output_path = f"./readability_res/integrated_readability_{dataset}_{retriever}_top_{_k}.csv"
try:
    exist_qids = pd.read_csv(output_path).qid.unique().values
except:
    exist_qids = []

for qid in tqdm(retr_res.qid.unique()):
    if(qid in exist_qids):
        continue
    df_content = []
    input = retr_res[(retr_res.qid==qid)&(retr_res['rank']<_k)]
    input_text = ''
    docno = f'{qid}_integrated_{_k}'
    
    for _t in text_loader(input).text.values:
        input_text += _t
    
    df_content += calculate_readability(input_text, qid, docno)


    temp_output = pd.DataFrame(df_content, columns=['qid', 'docno', 'readability_metric', 'score', 'grade'])

    csvfile = pathlib.Path(output_path)
    temp_output.to_csv(output_path, mode='a', index=False, header=not csvfile.exists())
    # print(temp_output)

100%|██████████| 3610/3610 [00:49<00:00, 73.29it/s]


In [75]:
readability_df = pd.DataFrame(df_content, columns=['qid', 'readability_metric', 'score', 'grade'])
readability_df.qid = readability_df.qid.apply(lambda x: str(int(x)))

ValueError: 4 columns passed, passed data had 5 columns

In [148]:
len(readability_df.qid.unique())

6980

In [151]:
import json
import numpy as np

_k = 3
_ret = 'e5'
_task = 'dl'
target_metric = 'f1'

_dataset_dev, _prefix, _suffix = 'dev_small', 'random', 'prompt1'

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../coherence_eval/log_prob_temp_res/full_context/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

dev_res = retr_res[['qid', 'query']].drop_duplicates().copy()
dev_res.qid = dev_res.qid.astype('str')
dev_res = dev_res[dev_res.qid.isin(readability_df.qid.unique())]

# Expand the dataframe for the convenience of analysis
for metric_name in readability_df.readability_metric.unique():
    value_dict = dict(zip(readability_df[readability_df.readability_metric==metric_name]['qid'], readability_df[readability_df.readability_metric==metric_name]['score']))
    dev_res[metric_name] = dev_res.qid.apply(lambda _qid: value_dict[_qid])

base_f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}
kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)

,qid,query,Dale Chall,Spache,Flesch-Kincaid,Flesch,Gunning Fog,Coleman Liau,ARI,Linsear Write,SMOG,f1,utility
0,1048585,what is paula deen s brother,12.501520,8.069801,10.033116,52.617554,11.247826,11.244928,10.233043,11.375000,-1.0,0.569778,0.272814
50,2,androgen receptor define,15.777106,16.727882,38.441274,-46.353328,43.884076,18.428790,45.540000,65.250000,-1.0,0.880292,0.251544
100,524332,treating tension headaches without medication,8.591167,6.202889,9.951481,49.356667,11.333333,12.976889,10.701556,8.944444,-1.0,0.536671,0.043722


In [174]:
from scipy import stats

a = {"Dale Chall": lambda: r.dale_chall(),
        "Spache": lambda: r.spache(),
        "Flesch-Kincaid": lambda: r.flesch_kincaid(),
        "Flesch": lambda: r.flesch(),
        "Gunning Fog": lambda: r.gunning_fog(),
        "Coleman Liau": lambda: r.coleman_liau(),
        "ARI": lambda: r.ari(),
        "Linsear Write": lambda: r.linsear_write(),
        "SMOG": lambda: r.smog()}

for check_metric_name in a.keys():

    print(check_metric_name)
    analyse_metric_df = dev_res[['qid', check_metric_name, 'f1', 'utility']]
    analyse_metric_df = analyse_metric_df[analyse_metric_df[check_metric_name]!=-1]
    
    print(stats.spearmanr(analyse_metric_df[check_metric_name], analyse_metric_df['f1']))
    print(stats.spearmanr(analyse_metric_df[check_metric_name], analyse_metric_df['utility']))

Dale Chall
SignificanceResult(statistic=np.float64(0.0980840617811912), pvalue=np.float64(1.6531995497496619e-15))
SignificanceResult(statistic=np.float64(0.07895173115665036), pvalue=np.float64(1.4956133787002233e-10))
Spache
SignificanceResult(statistic=np.float64(0.05219017682295496), pvalue=np.float64(2.328918504544953e-05))
SignificanceResult(statistic=np.float64(0.03023640080446905), pvalue=np.float64(0.014286075941371722))
Flesch-Kincaid
SignificanceResult(statistic=np.float64(0.04457083876400806), pvalue=np.float64(0.00030332256310342014))
SignificanceResult(statistic=np.float64(0.015246891883428752), pvalue=np.float64(0.2167519106678152))
Flesch
SignificanceResult(statistic=np.float64(-0.07690971773480935), pvalue=np.float64(4.3857247622271825e-10))
SignificanceResult(statistic=np.float64(-0.04171668070360087), pvalue=np.float64(0.0007224644291555767))
Gunning Fog
SignificanceResult(statistic=np.float64(0.02134065868109098), pvalue=np.float64(0.08381127371476883))
Significance